# CEPHA19 - multi-domain fine-tune (Aariz + ISBI 2015, shared 19)

Fine-tunes the 29-landmark Aariz model on the **19 shared landmarks** across two
radiographic domains, then exports ONNX + SSM prior and benchmarks the result.

- Fixed vocabulary: the 19 ISBI symbols (Aariz's 29 mapped onto them).
- Training data: Aariz train (averaged Junior+Senior) + ISBI 2015 train.
- Validation/monitoring: Aariz valid + ISBI test.
- Starts from the existing 29-landmark `best.pt` (backbone transferred; the
  19-channel head is retrained).

**Kaggle setup**
1. Accelerator: **GPU T4 x2**.
2. Internet: **On**.
3. **best.pt (only required input):** attach the output of your CEPHA29 training
   notebook via **Add Data -> Your Work -> Notebooks**, or upload `best.pt` as a
   dataset. Alternatively set `BEST_PT_URL` in the checkpoint cell.
4. **ISBI 2015 and Aariz are fetched automatically** (public mirror / Figshare);
   you do **not** need to attach them. Attaching is only a speed-up.

Outputs land in `/kaggle/working/deliverables_md19/`.

In [ ]:
!pip -q install timm albumentations opencv-python-headless pyyaml onnx onnxruntime pillow numpy scipy pandas requests
print("deps installed")

In [ ]:
import os
%cd /kaggle/working
!rm -rf CephaloHRNet CephalometryStudio
!git clone --depth 1 https://github.com/Cestovatels/CephaloHRNet.git
!git clone --depth 1 https://github.com/Muhammad-Nabeel-Sh/CephalometryStudio.git
assert os.path.exists("CephaloHRNet/models/hrnet.py")
assert os.path.exists("CephalometryStudio/scripts/ai/canonical.py"), "clone missing scripts/ai — push first"
print("cloned")

In [ ]:
import glob, os, sys, urllib.request
import numpy as np
sys.path.insert(0, "CephalometryStudio/scripts/ai")
from canonical import ISBI19, ISBI_TO_CEPHA

BASE = "https://raw.githubusercontent.com/stolariks/medical-landmark-detection/main/data/isbi-2015/"
ANN_TRAIN = "/kaggle/working/isbi_train.csv"
ANN_TEST = "/kaggle/working/isbi_test.csv"
for url, dest in [(BASE + "train/annotations.csv", ANN_TRAIN),
                  (BASE + "test/annotations.csv", ANN_TEST)]:
    if not os.path.exists(dest):
        req = urllib.request.Request(url, headers={"User-Agent": "python-urllib"})
        with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
            f.write(r.read())

def read_isbi_csv(path):
    import csv
    out = {}
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.reader(f):
            if not row or row[0] == "image_path":
                continue
            try:
                pts = np.array([float(v) for v in row[1:]], np.float64).reshape(-1, 2)
            except Exception:
                continue
            if pts.shape[0] == len(ISBI19):
                out[os.path.splitext(os.path.basename(row[0]))[0]] = pts
    return out

isbi_gt = {"train": read_isbi_csv(ANN_TRAIN), "test": read_isbi_csv(ANN_TEST)}
print("ISBI GT:", {k: len(v) for k, v in isbi_gt.items()})

def find_input_images(stems):
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        for p in glob.glob("/kaggle/input/**/" + ext, recursive=True):
            if os.path.splitext(os.path.basename(p))[0] in stems:
                return os.path.dirname(p)
    return None

ISBI_DIR = find_input_images(set(isbi_gt["train"]) | set(isbi_gt["test"]))
print("ISBI image dir:", ISBI_DIR or "(fetch from mirror)")

def isbi_image(split, stem):
    if ISBI_DIR:
        for ext in (".jpg", ".jpeg", ".png"):
            p = os.path.join(ISBI_DIR, stem + ext)
            if os.path.exists(p):
                return p
        return None
    cache = "/kaggle/working/isbi_img_" + split
    os.makedirs(cache, exist_ok=True)
    for ext in (".jpg", ".png"):
        dest = os.path.join(cache, stem + ext)
        if os.path.exists(dest):
            return dest
        try:
            url = BASE + split + "/cepha400/" + stem + ext
            req = urllib.request.Request(url, headers={"User-Agent": "python-urllib"})
            with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
                f.write(r.read())
            return dest
        except Exception:
            continue
    return None

isbi_cases = {"train": [], "test": []}
for split in ("train", "test"):
    for stem, gt in isbi_gt[split].items():
        p = isbi_image(split, stem)
        if p:
            isbi_cases[split].append((stem, p, gt))
    print("ISBI", split, "cases:", len(isbi_cases[split]))

def find_dataset(root):
    for base, dirs, _ in os.walk(root):
        for split in ("train", "valid"):
            if os.path.isdir(os.path.join(base, split, "Cephalograms")):
                return base
    return None

AARIZ = find_dataset("/kaggle/input")
if AARIZ is None:
    import zipfile
    URL = "https://ndownloader.figshare.com/files/51041642"
    ZIP = "/kaggle/working/Aariz.zip"
    if not os.path.exists(ZIP) or os.path.getsize(ZIP) != 2098209792:
        req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=900) as r, open(ZIP, "wb") as f:
            while True:
                c = r.read(1 << 20)
                if not c:
                    break
                f.write(c)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/kaggle/working/aariz")
    AARIZ = find_dataset("/kaggle/working/aariz")
print("AARIZ:", AARIZ)
assert AARIZ, "Aariz dataset not found"

In [ ]:
import glob, json, os, shutil
import numpy as np

MD = "/kaggle/working/md19"
EVAL = "/kaggle/working/eval"
for d in (MD, EVAL):
    shutil.rmtree(d, ignore_errors=True)

def ann_dir(root):
    return os.path.join(root, "Annotations", "Cephalometric Landmarks")

def read_syms(path):
    data = json.load(open(path, encoding="utf-8"))
    return {lm["symbol"]: (float(lm["value"]["x"]), float(lm["value"]["y"]))
            for lm in data.get("landmarks", []) if "x" in (lm.get("value") or {})}

def aariz_avg(split, stem):
    root = os.path.join(AARIZ, split, "Annotations", "Cephalometric Landmarks")
    dirs = [d for d in glob.glob(os.path.join(root, "*")) if os.path.isdir(d)]
    js = [os.path.join(d, stem + ".json") for d in dirs
          if os.path.exists(os.path.join(d, stem + ".json"))]
    if not js:
        return None
    sets = [read_syms(p) for p in js]
    keys = set(sets[0])
    for s in sets[1:]:
        keys &= set(s)
    return {k: (float(np.mean([s[k][0] for s in sets])),
                float(np.mean([s[k][1] for s in sets]))) for k in keys}

def write_case(root, prefix, domain, stem, img_path, pts):
    ceph = os.path.join(root, "Cephalograms")
    ad = ann_dir(root)
    os.makedirs(ceph, exist_ok=True)
    os.makedirs(ad, exist_ok=True)
    new_stem = (domain + "_" + stem) if prefix else stem
    ext = os.path.splitext(img_path)[1].lower()
    if ext not in (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"):
        ext = ".jpg"
    dst = os.path.join(ceph, new_stem + ext)
    if not os.path.exists(dst):
        shutil.copy(img_path, dst)
    doc = {"ceph_id": new_stem,
           "landmarks": [{"symbol": s, "value": {"x": pts[s][0], "y": pts[s][1]}} for s in ISBI19]}
    json.dump(doc, open(os.path.join(ad, new_stem + ".json"), "w"))

def aariz_pts(split, stem):
    avg = aariz_avg(split, stem)
    if not avg:
        return None
    if not all(ISBI_TO_CEPHA[s] in avg for s in ISBI19):
        return None
    return {s: avg[ISBI_TO_CEPHA[s]] for s in ISBI19}

for split in ("train", "valid"):
    for p in sorted(glob.glob(os.path.join(AARIZ, split, "Cephalograms", "*"))):
        stem = os.path.splitext(os.path.basename(p))[0]
        pts = aariz_pts(split, stem)
        if pts:
            write_case(os.path.join(MD, split), True, "aariz", stem, p, pts)
    src_split = "train" if split == "train" else "test"
    for stem, p, gt in isbi_cases[src_split]:
        pts = {s: tuple(gt[i]) for i, s in enumerate(ISBI19)}
        write_case(os.path.join(MD, split), True, "isbi", stem, p, pts)

for p in sorted(glob.glob(os.path.join(AARIZ, "valid", "Cephalograms", "*"))):
    stem = os.path.splitext(os.path.basename(p))[0]
    pts = aariz_pts("valid", stem)
    if pts:
        write_case(os.path.join(EVAL, "aariz19"), False, "aariz", stem, p, pts)
for stem, p, gt in isbi_cases["test"]:
    pts = {s: tuple(gt[i]) for i, s in enumerate(ISBI19)}
    write_case(os.path.join(EVAL, "isbi19"), False, "isbi", stem, p, pts)
if os.path.isdir(os.path.join(EVAL, "isbi19")):
    shutil.copy(ANN_TEST, os.path.join(EVAL, "isbi19", "annotations.csv"))

def count(root):
    return len(glob.glob(os.path.join(root, "Cephalograms", "*")))
print("md19 train:", count(os.path.join(MD, "train")),
      "valid:", count(os.path.join(MD, "valid")),
      "| eval aariz19:", count(os.path.join(EVAL, "aariz19")),
      "isbi19:", count(os.path.join(EVAL, "isbi19")))

In [ ]:
import re
# Patch CephaloHRNet for the 19-landmark shared vocabulary.
dp = "CephaloHRNet/data/dataset.py"
src = open(dp, encoding="utf-8").read()
new_block = "LANDMARK_SYMBOLS = [\n" + "".join("    %r,\n" % s for s in ISBI19) + "]"
src = re.sub(r"LANDMARK_SYMBOLS = \[.*?\]", new_block, src, count=1, flags=re.S)
src = re.sub(r"NUM_LANDMARKS\s*=\s*29", "NUM_LANDMARKS = 19", src, count=1)
open(dp, "w", encoding="utf-8").write(src)

tp = "CephaloHRNet/train.py"
src = open(tp, encoding="utf-8").read()
assert "num_landmarks=29" in src
open(tp, "w", encoding="utf-8").write(src.replace("num_landmarks=29", "num_landmarks=19"))
assert "NUM_LANDMARKS = 19" in open(dp, encoding="utf-8").read()
print("patched CephaloHRNet -> 19 landmarks:", ISBI19)

In [ ]:
import glob, os, urllib.request

BEST = None
pts = [p for p in glob.glob("/kaggle/input/**/*.pt", recursive=True)
       if os.path.basename(p) == "best.pt"]
if pts:
    BEST = sorted(pts, key=os.path.getmtime)[-1]

# Optional: fetch the checkpoint from a URL instead of attaching it.
BEST_PT_URL = ""  # e.g. "https://huggingface.co/<you>/<repo>/resolve/main/best.pt"
if BEST is None and BEST_PT_URL:
    dest = "/kaggle/working/best.pt"
    if not os.path.exists(dest):
        req = urllib.request.Request(BEST_PT_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=600) as r, open(dest, "wb") as f:
            while True:
                c = r.read(1 << 20)
                if not c:
                    break
                f.write(c)
    BEST = dest

print("best.pt:", BEST)
assert BEST, (
    "No 29-landmark best.pt found. Attach it via Add Data -> Your Work -> Notebooks "
    "-> your CEPHA29 training notebook (its output has runs/train/.../best.pt), or "
    "upload it as a Kaggle dataset, or set BEST_PT_URL. "
    "ISBI 2015 and Aariz are downloaded automatically and do NOT need attaching."
)

In [ ]:
import os, shutil
# Calibration images for INT8 quantization.
calib = "/kaggle/working/calib_md19"
os.makedirs(calib, exist_ok=True)
imgs = sorted(glob.glob(os.path.join(MD, "train", "Cephalograms", "*")))[:16]
for f in imgs:
    shutil.copy(f, os.path.join(calib, os.path.basename(f)))
print("calib:", len(imgs))

In [ ]:
import subprocess, sys
cmd = [sys.executable, "train.py",
       "--data", MD, "--model", "hrnet_w32",
       "--imgsz", "768", "--batch", "4", "--epochs", "120",
       "--loss", "awing", "--amp", "--device", "0",
       "--workers", "2", "--patience", "30",
       "--project", "runs/train", "--name", "md19_w32",
       "--annotators", "Merged19", "--img-dir", "Cephalograms",
       "--pretrained-path", BEST]
print(" ".join(cmd))
subprocess.run(cmd, cwd="/kaggle/working/CephaloHRNet", check=True)

In [ ]:
import glob, os, subprocess, sys
BESTR = sorted(glob.glob("/kaggle/working/CephaloHRNet/runs/train/md19_*/best.pt"),
               key=os.path.getmtime)[-1]
SAMPLE = sorted(glob.glob(os.path.join(MD, "train", "Cephalograms", "*")))[0]
print("weights:", BESTR)
subprocess.run([sys.executable, "CephalometryStudio/scripts/export-landmark-onnx.py",
                "--repo", "/kaggle/working/CephaloHRNet",
                "--weights", BESTR, "--model", "hrnet_w32",
                "--num-landmarks", "19", "--input-size", "768",
                "--out", "/kaggle/working/landmarks-md19.onnx",
                "--quantize", "--calib-dir", "/kaggle/working/calib_md19",
                "--verify", SAMPLE],
               cwd="/kaggle/working", check=True)

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, "CephalometryStudio/scripts/build-shape-model.py",
                "--set", "cepha19",
                "--ann-dir", os.path.join(MD, "train", "Annotations", "Cephalometric Landmarks", "Merged19"),
                "--k", "10",
                "--out", "/kaggle/working/shapeModel.cepha19.json"],
               cwd="/kaggle/working", check=True)

In [ ]:
import os, subprocess, sys
OUT = "/kaggle/working/audit_md19"
os.makedirs(OUT, exist_ok=True)
ERR = []

# 1) In-domain (Aariz valid, 19 shared) — raw, matching the app default.
r = subprocess.run([sys.executable, "CephalometryStudio/scripts/eval-landmark-model.py",
                    "--model", "/kaggle/working/landmarks-md19.int8.onnx",
                    "--set", "cepha19", "--limit", "200", "--dark",
                    "--ann-dir", os.path.join(EVAL, "aariz19", "Annotations", "Cephalometric Landmarks", "Merged19"),
                    "--images", os.path.join(EVAL, "aariz19", "Cephalograms"),
                    "--metrics-out", os.path.join(OUT, "aariz19_metrics.json")],
                   cwd="/kaggle/working")
if r.returncode: ERR.append("aariz")

# 2) External (ISBI test, held out from training) — raw, with production search.
r = subprocess.run([sys.executable, "CephalometryStudio/scripts/ai/external_benchmark.py",
                    "--model", "/kaggle/working/landmarks-md19.int8.onnx",
                    "--isbi-images", os.path.join(EVAL, "isbi19", "Cephalograms"),
                    "--isbi-ann", os.path.join(EVAL, "isbi19", "annotations.csv"),
                    "--search", "--no-ssm",
                    "--out", os.path.join(OUT, "isbi19")],
                   cwd="/kaggle/working")
if r.returncode: ERR.append("isbi")
print("failed:", ERR)

In [ ]:
import glob, hashlib, os, shutil
out = "/kaggle/working/deliverables_md19"
os.makedirs(out, exist_ok=True)
for name in ("landmarks-md19.onnx", "landmarks-md19.int8.onnx", "landmarks-md19-verify.png",
             "shapeModel.cepha19.json"):
    for p in glob.glob("/kaggle/working/" + name):
        shutil.copy(p, out)
for p in glob.glob("/kaggle/working/audit_md19/**/*", recursive=True):
    if os.path.isfile(p):
        rel = os.path.relpath(p, "/kaggle/working/audit_md19")
        dst = os.path.join(out, "audit", rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(p, dst)

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

print("=== DELIVERABLES ===")
for f in sorted(os.listdir(out)):
    p = os.path.join(out, f)
    if os.path.isfile(p):
        print(f"{os.path.getsize(p)/1e6:9.2f} MB  {f}\n            sha256 {sha(p)}")
print("(audit/ subfolder has the benchmark reports)")

## Next

1. Download `/kaggle/working/deliverables_md19/`.
2. Compare `audit/isbi19/isbi_metrics.json` against the baseline raw E0b
   (SDR@2 50.6%, median 1.97 mm). A successful fine-tune should **raise SDR@2**.
3. Compare `audit/aariz19_metrics.json` against the Aariz 29-model baseline
   (MRE ~2.55 mm, SDR@2 ~68.7%) on the shared 19 to check in-domain retention.
4. Inspect `landmarks-md19-verify.png` (19 channels, order = ISBI19).
5. If it improves, we wire it into the app as a new `cepha19` set.
6. Note: this run does **not** add flip/polarity/CLAHE augmentation yet — the
   first experiment isolates the effect of domain diversity. Augmentation is the
   next lever if this isn't enough.